# Análisis Exploratorio de Comunicación en WhatsApp

## Encuadre estadístico

Este notebook analiza un historial de conversación de WhatsApp desde una perspectiva estadística rigurosa, articulando los temas centrales del curso de posgrado **Tópicos de Estadística Avanzada**.

El objeto de estudio es una **serie temporal de eventos discretos**: cada mensaje es un evento caracterizado por un timestamp, un emisor y un contenido. A partir de esta estructura se construyen variables derivadas — tiempos de respuesta, longitudes de mensaje — cuya distribución, correlación y evolución temporal son el foco del análisis.

### Conexión con los temas del curso

| Sección del análisis | Tema del curso |
|---|---|
| Distribución de tiempos de respuesta | **Estimación puntual e intervalar** (media, mediana, IC bootstrap) |
| Comparación entre emisores | **Pruebas de hipótesis** no paramétricas (Mann-Whitney) |
| Evolución temporal de la actividad | **Series de tiempo**, tendencia y estacionalidad |
| Correlación longitud → demora | **Regresión** (MCO y transformaciones log) |
| Modelado de tiempos de respuesta | **Distribuciones de probabilidad** (exponencial, log-normal, Weibull) |

---

## 0. Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import re

warnings.filterwarnings('ignore')

# Estética general
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
PALETTE = ['#1e3c78', '#e05c2a']   # Azul UNS, naranja
sns.set_palette(PALETTE)
print('Entorno listo ✓')

## 1. Carga y parseo del archivo

WhatsApp exporta en formato argentino: `d/m/yyyy, HH:MM - Nombre: mensaje`. Algunos mensajes se extienden en múltiples líneas (continuaciones). El parser los une al mensaje anterior.

In [ ]:
# ── Ajustá este path al tuyo ──────────────────────────────────────────────────
CHAT_FILE = 'Chat_de_WhatsApp_con_Beatriz_Marron.txt'
# ─────────────────────────────────────────────────────────────────────────────

# Regex para líneas con timestamp (formato argentino d/m/yyyy o dd/mm/yyyy)
PATTERN = re.compile(
    r'^(\d{1,2}/\d{1,2}/\d{4}),\s(\d{1,2}:\d{2})\s-\s([^:]+):\s(.+)$'
)
SYSTEM_PATTERN = re.compile(
    r'^(\d{1,2}/\d{1,2}/\d{4}),\s(\d{1,2}:\d{2})\s-\s(?!.*:)'
)

records = []
with open(CHAT_FILE, encoding='utf-8') as f:
    for line in f:
        line = line.rstrip('\n')
        m = PATTERN.match(line)
        if m:
            records.append({
                'fecha_str': m.group(1),
                'hora_str':  m.group(2),
                'autor':     m.group(3).strip(),
                'mensaje':   m.group(4).strip()
            })
        elif records and not SYSTEM_PATTERN.match(line):
            # continuación de mensaje multilínea
            records[-1]['mensaje'] += ' ' + line.strip()

df = pd.DataFrame(records)

# Parseo de timestamp
df['timestamp'] = pd.to_datetime(
    df['fecha_str'] + ' ' + df['hora_str'],
    format='%d/%m/%Y %H:%M'
)
df = df.sort_values('timestamp').reset_index(drop=True)

# Filtrar mensajes de sistema y multimedia
EXCLUIR = ['<Multimedia omitido>', 'Los mensajes y las llamadas']
df = df[~df['mensaje'].str.contains('|'.join(EXCLUIR), na=False)].copy()
df = df[~df['mensaje'].str.endswith('.vcf (archivo adjunto)')].copy()
df = df.reset_index(drop=True)

# Variables temporales
df['año']       = df['timestamp'].dt.year
df['mes']       = df['timestamp'].dt.month
df['mes_nombre']= df['timestamp'].dt.strftime('%b')
df['dia_semana']= df['timestamp'].dt.dayofweek          # 0=lunes
df['dia_nombre']= df['timestamp'].dt.strftime('%a')
df['hora']      = df['timestamp'].dt.hour
df['fecha']     = df['timestamp'].dt.date
df['año_mes']   = df['timestamp'].dt.to_period('M')

# Longitud en palabras
df['n_palabras'] = df['mensaje'].str.split().str.len()

AUTORES = sorted(df['autor'].unique())
print(f'Mensajes cargados: {len(df):,}')
print(f'Período: {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'Autores: {AUTORES}')
df.head()

---
## 2. Exploración temporal: ¿cuándo se habla?

### 2.1 Actividad total por año

Comenzamos con la visión más agregada: ¿cómo evolucionó el volumen de comunicación a lo largo de los años?

In [ ]:
por_año = df.groupby(['año', 'autor']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Barras apiladas
por_año.plot(kind='bar', ax=axes[0], color=PALETTE, edgecolor='white', linewidth=0.5)
axes[0].set_title('Mensajes por año (por autor)')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Cantidad de mensajes')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(title='Autor')

# Total con línea de tendencia
total_año = por_año.sum(axis=1)
axes[1].bar(total_año.index, total_año.values, color='#1e3c78', alpha=0.7, edgecolor='white')
z = np.polyfit(range(len(total_año)), total_año.values, 1)
p = np.poly1d(z)
axes[1].plot(total_año.index, p(range(len(total_año))), 'r--', lw=1.5, label='Tendencia lineal')
axes[1].set_title('Total mensajes por año + tendencia')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Total mensajes')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

### 2.2 Actividad por mes del año (estacionalidad)

¿Hay meses del calendario en que se comunican más? Esto es análogo al análisis de **componente estacional** en series de tiempo.

In [ ]:
orden_meses = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
nombres_es  = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

por_mes = df.groupby(['mes', 'autor']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(12)
w = 0.35
for i, (autor, color) in enumerate(zip(por_mes.columns, PALETTE)):
    ax.bar(x + i*w - w/2, por_mes[autor].values, w, label=autor, color=color, alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(nombres_es)
ax.set_title('Mensajes por mes del año (acumulado todos los años)')
ax.set_ylabel('Cantidad de mensajes')
ax.legend(title='Autor')
plt.tight_layout()
plt.show()

### 2.3 Actividad por día de la semana

In [ ]:
dias_orden = [0,1,2,3,4,5,6]
dias_labels = ['Lun','Mar','Mié','Jue','Vie','Sáb','Dom']

por_dia = df.groupby(['dia_semana','autor']).size().unstack(fill_value=0)
por_dia = por_dia.reindex(dias_orden)

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(7)
w = 0.35
for i, (autor, color) in enumerate(zip(por_dia.columns, PALETTE)):
    ax.bar(x + i*w - w/2, por_dia[autor].values, w, label=autor, color=color, alpha=0.85, edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(dias_labels)
ax.set_title('Mensajes por día de la semana')
ax.set_ylabel('Cantidad de mensajes')
ax.legend(title='Autor')
plt.tight_layout()
plt.show()

### 2.4 Actividad por hora del día

La distribución horaria revela el **ritmo circadiano** de la comunicación. Es una distribución de probabilidad discreta sobre 24 categorías.

In [ ]:
por_hora = df.groupby(['hora','autor']).size().unstack(fill_value=0).reindex(range(24), fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Barras
x = np.arange(24)
w = 0.35
for i, (autor, color) in enumerate(zip(por_hora.columns, PALETTE)):
    axes[0].bar(x + i*w - w/2, por_hora[autor].values, w, label=autor, color=color, alpha=0.85, edgecolor='white')
axes[0].set_title('Mensajes por hora del día')
axes[0].set_xlabel('Hora')
axes[0].set_ylabel('Cantidad de mensajes')
axes[0].set_xticks(x[::2])
axes[0].legend(title='Autor')

# Heatmap hora × día de semana
pivot = df.groupby(['dia_semana','hora']).size().unstack(fill_value=0).reindex(columns=range(24), fill_value=0)
pivot.index = dias_labels
sns.heatmap(pivot, ax=axes[1], cmap='Blues', linewidths=0.3,
            cbar_kws={'label': 'Mensajes'})
axes[1].set_title('Heatmap: hora × día de semana')
axes[1].set_xlabel('Hora del día')
axes[1].set_ylabel('Día')

plt.tight_layout()
plt.show()

### 2.5 Serie temporal mensual

La serie mensual permite visualizar la **evolución completa** de la comunicación y aplicar técnicas de suavizado.

In [ ]:
serie_mensual = df.groupby('año_mes').size()
serie_mensual.index = serie_mensual.index.to_timestamp()

fig, ax = plt.subplots(figsize=(15, 4))
ax.fill_between(serie_mensual.index, serie_mensual.values, alpha=0.25, color='#1e3c78')
ax.plot(serie_mensual.index, serie_mensual.values, color='#1e3c78', lw=1.2, label='Mensajes/mes')

# Media móvil 6 meses
mm6 = serie_mensual.rolling(6, center=True).mean()
ax.plot(mm6.index, mm6.values, color='#e05c2a', lw=2, label='Media móvil 6 meses')

ax.set_title('Serie temporal mensual de mensajes (2016–2026)')
ax.set_xlabel('Período')
ax.set_ylabel('Mensajes')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Estadística descriptiva de los mensajes

### 3.1 Longitud de mensajes (en palabras)

La distribución de longitud suele ser **asimétrica positiva** (log-normal), un patrón recurrente en datos de comunicación humana.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histograma por autor
for autor, color in zip(AUTORES, PALETTE):
    sub = df[df['autor']==autor]['n_palabras'].clip(upper=60)
    axes[0].hist(sub, bins=30, alpha=0.6, color=color, label=autor, edgecolor='white')
axes[0].set_title('Distribución de longitud (palabras)')
axes[0].set_xlabel('Palabras por mensaje')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Escala log en eje x
for autor, color in zip(AUTORES, PALETTE):
    sub = df[df['autor']==autor]['n_palabras'].clip(lower=1)
    axes[1].hist(np.log(sub), bins=30, alpha=0.6, color=color, label=autor, edgecolor='white')
axes[1].set_title('Distribución de log(palabras)')
axes[1].set_xlabel('log(palabras)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

# Boxplot comparativo
data_box = [df[df['autor']==a]['n_palabras'].clip(upper=50).values for a in AUTORES]
bp = axes[2].boxplot(data_box, patch_artist=True, labels=AUTORES, widths=0.4)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title('Boxplot longitud (truncado en 50)')
axes[2].set_ylabel('Palabras por mensaje')

plt.tight_layout()
plt.show()

# Estadísticos
print('\nEstadísticos descriptivos — longitud (palabras):')
print(df.groupby('autor')['n_palabras'].describe().round(2))

---
## 4. Cálculo de tiempos de respuesta

### Marco conceptual

Definimos un **par de respuesta** como:
- Un turno de mensajes consecutivos del emisor $A$
- Seguido del **primer** mensaje del emisor $B$ antes de que $A$ vuelva a escribir

El tiempo de respuesta $T$ se mide desde el **último mensaje del turno de $A$** hasta el primer mensaje de respuesta de $B$.

Se excluyen tiempos $T = 0$ (mensajes con el mismo timestamp) y tiempos $T > 24$ horas (probablemente no son respuestas directas al turno anterior).

In [ ]:
def calcular_respuestas(df, autor_envia, autor_responde, max_horas=24):
    """Calcula pares de respuesta entre dos autores."""
    rows = []
    i = 0
    n = len(df)
    while i < n:
        if df.iloc[i]['autor'] == autor_envia:
            # Encontrar el último mensaje consecutivo de autor_envia
            j = i
            while j < n - 1 and df.iloc[j+1]['autor'] == autor_envia:
                j += 1
            # Buscar la próxima respuesta de autor_responde
            k = j + 1
            if k < n and df.iloc[k]['autor'] == autor_responde:
                t_envia   = df.iloc[j]['timestamp']
                t_responde = df.iloc[k]['timestamp']
                delta_min  = (t_responde - t_envia).total_seconds() / 60
                if 0 < delta_min <= max_horas * 60:
                    rows.append({
                        'emisor':       autor_envia,
                        'receptor':     autor_responde,
                        't_envio':      t_envia,
                        't_respuesta':  t_responde,
                        'demora_min':   delta_min,
                        'demora_seg':   delta_min * 60,
                        'año':          t_envia.year,
                        'hora_envio':   t_envia.hour,
                        'dia_semana':   t_envia.dayofweek,
                        'n_palabras_enviadas': df.iloc[j]['n_palabras'],
                        'n_palabras_respuesta': df.iloc[k]['n_palabras'],
                    })
            i = j + 1
        else:
            i += 1
    return pd.DataFrame(rows)

A1, A2 = AUTORES[0], AUTORES[1]

resp_A1_A2 = calcular_respuestas(df, A1, A2)  # A1 envía → A2 responde
resp_A2_A1 = calcular_respuestas(df, A2, A1)  # A2 envía → A1 responde
resp_all   = pd.concat([resp_A1_A2, resp_A2_A1], ignore_index=True)

print(f'Pares de respuesta {A1} → {A2}: {len(resp_A1_A2)}')
print(f'Pares de respuesta {A2} → {A1}: {len(resp_A2_A1)}')
print(f'Total: {len(resp_all)}')
resp_all.head()

---
## 5. Distribución de tiempos de respuesta

### Conexión con el curso: Estimación puntual e intervalar

La distribución de tiempos de respuesta es un ejemplo canónico de variable aleatoria positiva asimétrica. Para su análisis aplicamos:

- **Estimadores de posición**: media $\bar{T}$ y mediana $\tilde{T}$. La mediana es más robusta ante outliers (respondí a las 3am).
- **Estimadores de dispersión**: desvío estándar y rango intercuartílico (RIC).
- **Intervalos de confianza bootstrap**: para la mediana, sin asumir normalidad.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# ── Histograma escala natural ──────────────────────────────────────────────
for sub, color, label in [
    (resp_A1_A2, PALETTE[0], f'{A1} → {A2}'),
    (resp_A2_A1, PALETTE[1], f'{A2} → {A1}'),
]:
    axes[0,0].hist(sub['demora_min'].clip(upper=120), bins=40,
                   alpha=0.55, color=color, label=label, edgecolor='white')
axes[0,0].set_title('Distribución de demora (min) — hasta 2 horas')
axes[0,0].set_xlabel('Minutos')
axes[0,0].set_ylabel('Frecuencia')
axes[0,0].legend()

# ── Histograma log ─────────────────────────────────────────────────────────
for sub, color, label in [
    (resp_A1_A2, PALETTE[0], f'{A1} → {A2}'),
    (resp_A2_A1, PALETTE[1], f'{A2} → {A1}'),
]:
    axes[0,1].hist(np.log1p(sub['demora_min']), bins=40,
                   alpha=0.55, color=color, label=label, edgecolor='white')
axes[0,1].set_title('Distribución de log(1+demora)')
axes[0,1].set_xlabel('log(1 + minutos)')
axes[0,1].set_ylabel('Frecuencia')
axes[0,1].legend()

# ── ECDF ───────────────────────────────────────────────────────────────────
for sub, color, label in [
    (resp_A1_A2, PALETTE[0], f'{A1} → {A2}'),
    (resp_A2_A1, PALETTE[1], f'{A2} → {A1}'),
]:
    datos = np.sort(sub['demora_min'].values)
    ecdf  = np.arange(1, len(datos)+1) / len(datos)
    axes[1,0].plot(datos, ecdf, color=color, lw=1.8, label=label)
axes[1,0].axvline(30, color='gray', ls='--', lw=1, label='30 min')
axes[1,0].axvline(60, color='silver', ls='--', lw=1, label='1 hora')
axes[1,0].set_xlim(0, 200)
axes[1,0].set_title('Función de distribución empírica (ECDF)')
axes[1,0].set_xlabel('Minutos')
axes[1,0].set_ylabel('F(t)')
axes[1,0].legend(fontsize=8)

# ── Boxplot comparativo ────────────────────────────────────────────────────
data_box  = [resp_A1_A2['demora_min'].clip(upper=300).values,
             resp_A2_A1['demora_min'].clip(upper=300).values]
labels_bp = [f'{A1}→{A2}', f'{A2}→{A1}']
bp = axes[1,1].boxplot(data_box, patch_artist=True, labels=labels_bp, widths=0.4)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1,1].set_title('Boxplot demora (min) — truncado en 300')
axes[1,1].set_ylabel('Minutos')

plt.suptitle('Tiempos de respuesta', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Tabla de estadísticos ──────────────────────────────────────────────────
def resumen_demora(df_resp, nombre):
    d = df_resp['demora_min']
    # IC bootstrap para la mediana
    np.random.seed(42)
    boot = [np.median(np.random.choice(d, len(d), replace=True)) for _ in range(2000)]
    ic_lo, ic_hi = np.percentile(boot, [2.5, 97.5])
    return pd.Series({
        'N pares':         len(d),
        'Media (min)':     d.mean().round(1),
        'Mediana (min)':   d.median().round(1),
        'IC 95% mediana':  f'[{ic_lo:.1f}, {ic_hi:.1f}]',
        'Desvío est.':     d.std().round(1),
        'RIC':             f'[{d.quantile(.25):.1f}, {d.quantile(.75):.1f}]',
        'P10 (min)':       d.quantile(.10).round(1),
        'P90 (min)':       d.quantile(.90).round(1),
    }, name=nombre)

tabla = pd.concat([
    resumen_demora(resp_A1_A2, f'{A1} → {A2}'),
    resumen_demora(resp_A2_A1, f'{A2} → {A1}'),
], axis=1)
print('\n=== Estadísticos de tiempos de respuesta ===')
print(tabla.to_string())

### Ajuste de distribuciones paramétricas

**Conexión con el curso: Distribuciones de probabilidad**

Los tiempos de espera entre eventos en un proceso de Poisson siguen una distribución **Exponencial**. Cuando los tiempos son sumas de varias etapas independientes (leer, pensar, escribir), emergen distribuciones **log-normal** o **Gamma**. Ajustamos ambas y comparamos con el test de Kolmogorov-Smirnov.

In [ ]:
from scipy.stats import kstest, expon, lognorm, gamma

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (df_resp, titulo) in zip(axes, [
    (resp_A1_A2, f'{A1} → {A2}'),
    (resp_A2_A1, f'{A2} → {A1}'),
]):
    data = df_resp['demora_min'].clip(lower=0.1).values
    x_range = np.linspace(0.1, np.percentile(data, 95), 300)

    ax.hist(data[data <= np.percentile(data,95)], bins=40, density=True,
            alpha=0.4, color='steelblue', label='Datos', edgecolor='white')

    # Exponencial
    loc_e, scale_e = expon.fit(data, floc=0)
    ax.plot(x_range, expon.pdf(x_range, loc_e, scale_e),
            'r-', lw=1.8, label='Exponencial')
    ks_e = kstest(data, 'expon', args=(loc_e, scale_e))

    # Log-normal
    s_l, loc_l, scale_l = lognorm.fit(data, floc=0)
    ax.plot(x_range, lognorm.pdf(x_range, s_l, loc_l, scale_l),
            'g-', lw=1.8, label='Log-normal')
    ks_l = kstest(data, 'lognorm', args=(s_l, loc_l, scale_l))

    # Gamma
    a_g, loc_g, scale_g = gamma.fit(data, floc=0)
    ax.plot(x_range, gamma.pdf(x_range, a_g, loc_g, scale_g),
            'm--', lw=1.5, label='Gamma')
    ks_g = kstest(data, 'gamma', args=(a_g, loc_g, scale_g))

    ax.set_title(f'{titulo}\n'
                 f'KS — Exp: {ks_e.statistic:.3f} (p={ks_e.pvalue:.3f}), '
                 f'LogN: {ks_l.statistic:.3f} (p={ks_l.pvalue:.3f}), '
                 f'Gamma: {ks_g.statistic:.3f} (p={ks_g.pvalue:.3f})',
                 fontsize=9)
    ax.set_xlabel('Minutos')
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=8)

plt.suptitle('Ajuste de distribuciones paramétricas a tiempos de respuesta', fontsize=12)
plt.tight_layout()
plt.show()

---
## 6. Prueba de hipótesis: ¿responden igual de rápido?

### Conexión con el curso: Pruebas de hipótesis

**Hipótesis:**
$$H_0: F_{T_A}(t) = F_{T_B}(t) \quad \forall t \quad \text{(misma distribución de demora)}$$
$$H_1: F_{T_A}(t) \neq F_{T_B}(t)$$

Dado que los tiempos de respuesta son claramente no normales (asimetría positiva), usamos la prueba **Mann-Whitney U** (no paramétrica). Esta prueba evalúa si los tiempos de un emisor tienden a ser sistemáticamente menores o mayores que los del otro.

In [ ]:
from scipy.stats import mannwhitneyu, shapiro

d1 = resp_A1_A2['demora_min'].values
d2 = resp_A2_A1['demora_min'].values

stat_mw, pval_mw = mannwhitneyu(d1, d2, alternative='two-sided')

# Tamaño del efecto: r = Z / sqrt(N)
N = len(d1) + len(d2)
Z = stats.norm.ppf(pval_mw/2) if pval_mw < 1 else 0
r_effect = abs(Z) / np.sqrt(N)

print('=== Test Mann-Whitney U ===')
print(f'Estadístico U:  {stat_mw:.1f}')
print(f'p-valor:        {pval_mw:.4f}')
print(f'Tamaño efecto r: {r_effect:.3f}  (pequeño<0.1, mediano≈0.3, grande>0.5)')
print()
print(f'Mediana {A1} → {A2}: {np.median(d1):.1f} min')
print(f'Mediana {A2} → {A1}: {np.median(d2):.1f} min')

if pval_mw < 0.05:
    print('\n→ Se rechaza H₀ (α=0.05): las distribuciones de demora difieren significativamente.')
else:
    print('\n→ No se rechaza H₀ (α=0.05): no hay evidencia de diferencia significativa.')

---
## 7. ¿Los mensajes más largos tardan más en responderse?

### Conexión con el curso: Regresión lineal simple

Ajustamos el modelo:
$$\log(T_i + 1) = \beta_0 + \beta_1 \cdot \text{palabras}_i + \varepsilon_i$$

La transformación logarítmica en la variable respuesta $T$ es necesaria para estabilizar la varianza y acercar la distribución de residuos a la normalidad (supuesto del modelo lineal). Interpretamos $\hat{\beta}_1$: un mensaje con una palabra extra está asociado a un cambio de $\hat{\beta}_1$ unidades en $\log(T+1)$.

In [ ]:
from scipy.stats import pearsonr, spearmanr

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df_resp, titulo, color) in zip(axes, [
    (resp_A1_A2, f'{A1} → {A2}', PALETTE[0]),
    (resp_A2_A1, f'{A2} → {A1}', PALETTE[1]),
]):
    x = df_resp['n_palabras_enviadas'].clip(upper=50)
    y = np.log1p(df_resp['demora_min'])

    ax.scatter(x, y, alpha=0.25, s=18, color=color, edgecolors='none')

    # Regresión
    z = np.polyfit(x, y, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, p(x_line), 'k-', lw=2, label=f'β₁={z[0]:.3f}')

    rho, p_sp = spearmanr(x, y)
    r, p_pe   = pearsonr(x, y)

    ax.set_title(f'{titulo}\n'
                 f'Spearman ρ={rho:.3f} (p={p_sp:.3f})  |  Pearson r={r:.3f} (p={p_pe:.3f})')
    ax.set_xlabel('Palabras en el mensaje enviado')
    ax.set_ylabel('log(1 + demora en minutos)')
    ax.legend()

plt.suptitle('Correlación: longitud del mensaje → demora en responder', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot de demora por bins de longitud
resp_all['bin_palabras'] = pd.cut(
    resp_all['n_palabras_enviadas'],
    bins=[0, 3, 7, 15, 30, 200],
    labels=['1–3', '4–7', '8–15', '16–30', '31+']
)

fig, ax = plt.subplots(figsize=(10, 5))
grupos = [resp_all[resp_all['bin_palabras']==b]['demora_min'].clip(upper=300).values
          for b in ['1–3','4–7','8–15','16–30','31+']]
bp = ax.boxplot(grupos, patch_artist=True, labels=['1–3','4–7','8–15','16–30','31+'], widths=0.45)
colors_grad = ['#c8d8f0','#91b2e0','#5a8cd0','#1e5cb0','#0f2e78']
for patch, color in zip(bp['boxes'], colors_grad):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_xlabel('Palabras en el mensaje enviado')
ax.set_ylabel('Demora en responder (min)')
ax.set_title('Demora según longitud del mensaje enviado (ambos autores)')
plt.tight_layout()
plt.show()

---
## 8. Evolución temporal de la demora

¿La demora en responder cambió a lo largo de los años? Calculamos la mediana anual de la demora para cada dirección.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Mediana de demora por año
mediana_año = resp_all.groupby(['año','emisor'])['demora_min'].median().unstack()
for col, color in zip(mediana_año.columns, PALETTE):
    axes[0].plot(mediana_año.index, mediana_año[col], marker='o',
                 color=color, lw=2, label=f'{col} enviando')
axes[0].set_title('Mediana de demora por año')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Mediana demora (min)')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=45)

# Demora mediana por hora de envío
dem_hora = resp_all.groupby(['hora_envio','emisor'])['demora_min'].median().unstack().reindex(range(24))
for col, color in zip(dem_hora.columns, PALETTE):
    axes[1].plot(dem_hora.index, dem_hora[col], marker='o', ms=5,
                 color=color, lw=1.8, label=f'{col} enviando')
axes[1].set_title('Mediana de demora según hora del envío')
axes[1].set_xlabel('Hora del envío')
axes[1].set_ylabel('Mediana demora (min)')
axes[1].set_xticks(range(0, 24, 2))
axes[1].set_xlim(-0.5, 23.5)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 9. ¿Los mensajes largos tardan más en *escribirse*?

Segunda hipótesis: ¿B tarda más en escribir respuestas largas? Aquí la variable predictora es la longitud de la *respuesta* (no el mensaje recibido), y la variable respuesta sigue siendo la demora.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df_resp, titulo, color) in zip(axes, [
    (resp_A1_A2, f'Demora de {A2} según longitud de su respuesta', PALETTE[1]),
    (resp_A2_A1, f'Demora de {A1} según longitud de su respuesta', PALETTE[0]),
]):
    x = df_resp['n_palabras_respuesta'].clip(upper=50)
    y = np.log1p(df_resp['demora_min'])

    ax.scatter(x, y, alpha=0.25, s=18, color=color, edgecolors='none')
    z = np.polyfit(x, y, 1)
    p_fit = np.poly1d(z)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, p_fit(x_line), 'k-', lw=2, label=f'β₁={z[0]:.3f}')

    rho, p_sp = spearmanr(x, y)
    ax.set_title(f'{titulo}\nSpearman ρ={rho:.3f} (p={p_sp:.3f})')
    ax.set_xlabel('Palabras en la respuesta')
    ax.set_ylabel('log(1 + demora en minutos)')
    ax.legend()

plt.suptitle('¿Se tarda más en escribir respuestas largas?', fontsize=12)
plt.tight_layout()
plt.show()

---
## 10. Síntesis y discusión estadística

### Resumen de hallazgos

In [ ]:
print('='*60)
print('SÍNTESIS DEL ANÁLISIS')
print('='*60)

for df_resp, label in [(resp_A1_A2, f'{A1} → {A2}'), (resp_A2_A1, f'{A2} → {A1}')]:
    d = df_resp['demora_min']
    x = df_resp['n_palabras_enviadas'].clip(upper=50)
    y = np.log1p(d)
    rho, p_sp = spearmanr(x, y)
    print(f'\n[{label}]')
    print(f'  N pares de respuesta:  {len(d)}')
    print(f'  Mediana demora:        {d.median():.1f} min ({d.median()/60:.2f} horas)')
    print(f'  Media demora:          {d.mean():.1f} min')
    print(f'  RIC:                   [{d.quantile(.25):.1f}, {d.quantile(.75):.1f}] min')
    print(f'  P90:                   {d.quantile(.9):.1f} min')
    print(f'  Corr. Spearman (long→demora): ρ={rho:.3f}, p={p_sp:.3f}')

print(f'\n[Prueba Mann-Whitney: ¿responden igual de rápido?]')
print(f'  U={stat_mw:.1f}, p={pval_mw:.4f}')
if pval_mw < 0.05:
    print('  → Diferencia estadísticamente significativa (α=0.05)')
else:
    print('  → Sin diferencia significativa (α=0.05)')
print('='*60)

### Reflexión final: lo que los datos de WhatsApp enseñan sobre estadística

Este dataset ilustra varios principios estadísticos fundamentales del curso:

1. **Sesgo de la media vs. robustez de la mediana.** Los tiempos de respuesta tienen outliers extremos (mensajes sin respuesta inmediata, horario nocturno). La media es sensible a ellos; la mediana con IC bootstrap es el estimador natural.

2. **Transformaciones estabilizadoras de varianza.** La distribución de $T$ es asimétrica positiva: $\log(T+1)$ es aproximadamente normal, habilitando el uso de regresión lineal clásica con sus supuestos.

3. **Elección de prueba según distribución.** Mann-Whitney U en lugar de $t$-test de dos muestras, porque la normalidad es claramente rechazada (Shapiro-Wilk).

4. **Correlación de Spearman vs. Pearson.** Para variables con distribución no normal y posibles outliers, el coeficiente de Spearman es más apropiado que el de Pearson.

5. **Ajuste de distribuciones y test KS.** La comparación Exponencial / Log-normal / Gamma ilustra el flujo hipotético-deductivo: proponer un modelo generativo, estimar sus parámetros por MV (máxima verosimilitud), y evaluar el ajuste con una prueba de bondad.

6. **Series de tiempo y estacionalidad.** La actividad mensual tiene componentes de tendencia (¿años más activos?) y estacionales (¿meses académicos?).

---
## 11. Estimación bayesiana de la tasa de respuesta

### Conexión con el curso: Estimación bayesiana

En la sección anterior estimamos los tiempos de respuesta desde una perspectiva **frecuentista**: calculamos la mediana muestral y construimos intervalos de confianza bootstrap.

Ahora adoptamos el enfoque **bayesiano**. El modelo generativo es:

$$T_i \mid \lambda \sim \text{Exponencial}(\lambda), \quad i = 1, \ldots, n$$

donde $\lambda > 0$ es la **tasa de respuesta** (respuestas por minuto). La media del tiempo de respuesta es $\mathbb{E}[T] = 1/\lambda$.

#### Prior conjugado

Elegimos un prior de la familia **Gamma**, que es conjugado de la exponencial:

$$\lambda \sim \text{Gamma}(\alpha_0, \beta_0)$$

Con $\alpha_0 = 1$, $\beta_0 = 1$ se obtiene un prior poco informativo (equivalente a haber observado 1 respuesta con tiempo acumulado de 1 minuto).

#### Posterior analítico

Dada la conjugación, el posterior es exacto y no requiere MCMC:

$$\lambda \mid T_1, \ldots, T_n \sim \text{Gamma}\!\left(\alpha_0 + n,\; \beta_0 + \sum_{i=1}^n T_i\right)$$

La **media posterior** es $\hat{\lambda}_{\text{Bayes}} = \dfrac{\alpha_0 + n}{\beta_0 + \sum T_i}$, que se acerca al estimador de máxima verosimilitud $\hat{\lambda}_{\text{MV}} = n / \sum T_i$ a medida que $n$ crece (el prior se *lava* con los datos).

El **intervalo de credibilidad** al 95% es el intervalo que contiene el 95% de la masa posterior — interpretable directamente como probabilidad sobre $\lambda$, a diferencia del IC frecuentista.

In [ ]:
from scipy.stats import gamma as gamma_dist

# Prior poco informativo: Gamma(1, 1)
alpha0, beta0 = 1.0, 1.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df_resp, titulo, color) in zip(axes, [
    (resp_A1_A2, f'{A1} → {A2}', PALETTE[0]),
    (resp_A2_A1, f'{A2} → {A1}', PALETTE[1]),
]):
    T = df_resp['demora_min'].values
    n = len(T)
    suma_T = T.sum()

    # Parámetros posterior
    alpha_post = alpha0 + n
    beta_post  = beta0 + suma_T

    # Rango de lambda para graficar
    lam_max = gamma_dist.ppf(0.9995, alpha_post, scale=1/beta_post)
    lam_range = np.linspace(1e-6, lam_max, 600)

    # Prior y posterior
    prior_pdf = gamma_dist.pdf(lam_range, alpha0, scale=1/beta0)
    post_pdf  = gamma_dist.pdf(lam_range, alpha_post, scale=1/beta_post)

    ax.plot(lam_range, prior_pdf, 'gray', lw=1.5, ls='--', label=f'Prior Gamma({alpha0:.0f},{beta0:.0f})')
    ax.fill_between(lam_range, post_pdf, alpha=0.25, color=color)
    ax.plot(lam_range, post_pdf, color=color, lw=2, label=f'Posterior Gamma({alpha_post:.0f},{beta_post:.0f})')

    # MV y media posterior
    lam_mv   = n / suma_T
    lam_mean = alpha_post / beta_post
    ic_lo    = gamma_dist.ppf(0.025, alpha_post, scale=1/beta_post)
    ic_hi    = gamma_dist.ppf(0.975, alpha_post, scale=1/beta_post)

    ax.axvline(lam_mv,   color='black',  ls=':',  lw=1.5, label=f'MV λ={lam_mv:.4f}')
    ax.axvline(lam_mean, color=color,    ls='-',  lw=1.5, label=f'Media post. λ={lam_mean:.4f}')
    ax.axvspan(ic_lo, ic_hi, alpha=0.12, color=color, label=f'IC 95% cred. [{ic_lo:.4f},{ic_hi:.4f}]')

    ax.set_title(
        f'{titulo}  (n={n})\n'
        f'Media posterior 1/λ = {1/lam_mean:.1f} min  |  '
        f'IC cred. 95%: [{1/ic_hi:.1f}, {1/ic_lo:.1f}] min',
        fontsize=9
    )
    ax.set_xlabel('λ (tasa de respuesta, resp/min)')
    ax.set_ylabel('Densidad posterior')
    ax.legend(fontsize=7.5)

plt.suptitle('Estimación bayesiana de λ — modelo Exponencial con prior Gamma conjugado',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Comparación frecuentista vs. bayesiano ─────────────────────────────
print('\n=== Frecuentista vs. Bayesiano ===')
for df_resp, label in [(resp_A1_A2, f'{A1}→{A2}'), (resp_A2_A1, f'{A2}→{A1}')]:
    T = df_resp['demora_min'].values
    n, suma_T = len(T), T.sum()
    alpha_p = alpha0 + n
    beta_p  = beta0 + suma_T
    lam_mv  = n / suma_T
    lam_mp  = alpha_p / beta_p
    ic_lo   = gamma_dist.ppf(0.025, alpha_p, scale=1/beta_p)
    ic_hi   = gamma_dist.ppf(0.975, alpha_p, scale=1/beta_p)
    # IC frecuentista para media (via bootstrap)
    np.random.seed(42)
    boot_medias = [np.mean(np.random.choice(T, n, replace=True)) for _ in range(2000)]
    freq_lo, freq_hi = np.percentile(boot_medias, [2.5, 97.5])
    print(f'\n[{label}]  n={n}')
    print(f'  MV  1/λ (media estimada):  {1/lam_mv:.2f} min')
    print(f'  Bayes media posterior 1/λ: {1/lam_mp:.2f} min')
    print(f'  IC 95% frecuentista (bootstrap media): [{freq_lo:.2f}, {freq_hi:.2f}] min')
    print(f'  IC 95% credibilidad (en 1/λ):          [{1/ic_hi:.2f}, {1/ic_lo:.2f}] min')

---
## 12. Nube de palabras

Una representación visual del vocabulario del chat. El tamaño de cada palabra es proporcional a su frecuencia. Se filtran stopwords del español y palabras propias del formato de WhatsApp.

In [ ]:
from wordcloud import WordCloud, STOPWORDS

STOPWORDS_ES = set(STOPWORDS) | {
    'que', 'de', 'en', 'el', 'la', 'los', 'las', 'un', 'una', 'unos', 'unas',
    'y', 'o', 'a', 'es', 'se', 'por', 'con', 'no', 'lo', 'le', 'si', 'me',
    'te', 'ya', 'al', 'del', 'su', 'sus', 'para', 'pero', 'más', 'como',
    'está', 'estoy', 'estamos', 'tienen', 'tiene', 'hay', 'ser', 'ser',
    'este', 'esta', 'esto', 'ese', 'esa', 'eso', 'mi', 'tu', 'vos', 'yo',
    'nos', 'bien', 'todo', 'todos', 'ok', 'OK', 'sí', 'si', 'también',
    'multimedia', 'omitido', 'mensaje', 'adjunto', 'cuando', 'porque',
    'qué', 'cómo', 'dónde', 'quién', 'cuál', 'así', 'aunque', 'vez',
    'le', 'les', 'son', 'fue', 'era', 'muy', 'han', 'he', 'ha', 'hoy',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

configs = [
    (df, 'Todos los mensajes', '#1e3c78', 'white'),
    (df[df['autor']==A1], A1, '#1e3c78', '#eef3fc'),
    (df[df['autor']==A2], A2, '#e05c2a', '#fff4ee'),
]

for ax, (df_sub, titulo, color_wc, bg) in zip(axes, configs):
    texto = ' '.join(df_sub['mensaje'].dropna().astype(str).str.lower())
    wc = WordCloud(
        width=800, height=500,
        background_color=bg,
        stopwords=STOPWORDS_ES,
        color_func=lambda *args, **kwargs: color_wc,
        max_words=120,
        collocations=False,
        min_word_length=3,
    ).generate(texto)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(titulo, fontsize=12, fontweight='bold', pad=10)

plt.suptitle('Nube de palabras — chat completo 2016–2026', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top 20 palabras por autor
from collections import Counter
import re as _re

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (autor, color) in zip(axes, zip(AUTORES, PALETTE)):
    texto = ' '.join(df[df['autor']==autor]['mensaje'].dropna().astype(str).str.lower())
    palabras = [w for w in _re.findall(r'[a-záéíóúüñ]+', texto)
                if w not in STOPWORDS_ES and len(w) >= 3]
    top20 = Counter(palabras).most_common(20)
    words_top, counts_top = zip(*top20)
    y_pos = range(len(words_top))
    ax.barh(list(y_pos), counts_top, color=color, alpha=0.8, edgecolor='white')
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(words_top)
    ax.invert_yaxis()
    ax.set_title(f'Top 20 palabras — {autor}')
    ax.set_xlabel('Frecuencia')

plt.tight_layout()
plt.show()

<br><br>
<div style="text-align:center; padding: 40px 60px; border-top: 2px solid #1e3c78;">
<h2 style="color:#1e3c78; font-size:1.6em; line-height:1.7; font-weight:400;">
<em>Aprender estadística es aprender a escuchar lo que los datos tienen para decir.<br>
Un lenguaje pronunciado en cada observación recolectada, traduciendo fenómenos muy diversos.<br>
La herramienta siempre es la misma. La curiosidad, también.</em>
</h2>
</div>